# SeeSaw AI: train your own detector (guns, fire and smoke, PPE)

Run this notebook on Google Colab with a GPU runtime (Runtime > Change runtime type > T4 GPU).
It produces two files, `<name>.onnx` and `<name>.names`, that you copy into the `models/` folder of SeeSaw AI.
The matching feature switches on by itself after a server restart.

1. Pick a dataset on [Roboflow Universe](https://universe.roboflow.com): search **gun detection** (or *weapon detection*, *fire and smoke detection*, *hard hat / PPE detection*).
2. On the dataset page click **Download dataset**, choose the **YOLOv8** format and **show download code**. Paste the workspace, project and version into the cell below, together with your Roboflow API key.
3. Set `MODEL_NAME` to `weapon`, `fire_smoke` or `ppe` and run every cell.

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
MODEL_NAME = "weapon"        # weapon, fire_smoke or ppe
EPOCHS = 50                  # 50 is a good first run; 100 or more for the final model
IMGSZ = 640

from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("PASTE_WORKSPACE").project("PASTE_PROJECT")
dataset = project.version(1).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print(open(DATA_YAML).read())

In [ ]:
!yolo train model=yolov8n.pt data={DATA_YAML} epochs={EPOCHS} imgsz={IMGSZ} project=runs name={MODEL_NAME} exist_ok=True

In [ ]:
# accuracy on the validation set: look at mAP50 per class
!yolo val model=runs/{MODEL_NAME}/weights/best.pt data={DATA_YAML} imgsz={IMGSZ}

In [ ]:
import shutil, yaml
!yolo export model=runs/{MODEL_NAME}/weights/best.pt format=onnx opset=12 imgsz={IMGSZ}
names = yaml.safe_load(open(DATA_YAML))["names"]
names = list(names.values()) if isinstance(names, dict) else list(names)
shutil.copy(f"runs/{MODEL_NAME}/weights/best.onnx", f"{MODEL_NAME}.onnx")
open(f"{MODEL_NAME}.names", "w").write("\n".join(names) + "\n")
print("classes:", names)
from google.colab import files
files.download(f"{MODEL_NAME}.onnx")
files.download(f"{MODEL_NAME}.names")

## Install on the PC

Copy both downloaded files into `models/` next to `object_detection_yolox_2022nov.onnx`, restart the server, and the
feature (Gun / Knife, Fire / Smoke or PPE) changes from "needs model file" to active. For PPE the alert fires on classes
whose names start with `no-`, `without` or `missing`, for example `NO-Hardhat`; for weapons and fire every detection alerts.